In [ ]:
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from transformers import AutoTokenizer

# NLTK 필수 리소스 다운로드
nltk.download('punkt')
# 최신 NLTK 버전 패치 대응 (punkt_tab 관련 에러 예방용)
nltk.download('punkt_tab')


english_text = "Natural Language Processing is fascinating. We are learning Subword Tokenization!"

# ① 문장 토큰화 (Sentence Tokenization)
english_sentences = sent_tokenize(english_text)
print("--- 1. 영어 문장 토큰화 ---")
print(english_sentences)

# ② 단어 토큰화 (Word Tokenization)
english_words = word_tokenize(english_text)
print("\n--- 2. 영어 단어 토큰화 ---")
print(english_words)

# ③ 형태소 토큰화 (Morpheme Tokenization)
# 영어는 형태소 분석 대신 단어 어근 추출(Stemming) 방식 활용
from nltk.stem import PorterStemmer
ps = PorterStemmer()
english_morphs = [ps.stem(word) for word in english_words]
print("\n--- 3. 영어 형태소(어근) 토큰화 ---")
print(english_morphs)

# ④ 서브워드 토큰화 (Subword Tokenization - BERT Tokenizer)
tokenizer_en = AutoTokenizer.from_pretrained("bert-base-uncased")
english_subwords = tokenizer_en.tokenize(english_text)
print("\n--- 4. 영어 서브워드 토큰화 ---")
print(english_subwords)

: 

In [ ]:

from konlpy.tag import Okt
import os

# Hugging Face 심볼릭 링크 경고 끌어오기
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
from transformers import AutoTokenizer

korean_text = "자연어 처리는 정말 재미있습니다. 우리는 서브워드 토큰화를 배웁니다!"

# ① 문장 토큰화 (Sentence Tokenization)
# 마침표/문맥 기반 분할
import re
korean_sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', korean_text)]
print("--- 1. 한국어 문장 토큰화 ---")
print(korean_sentences)

# ② 단어(어절) 토큰화 (Word Tokenization)
korean_words = korean_text.split()
print("\n--- 2. 한국어 단어(어절) 토큰화 ---")
print(korean_words)

# ③ 형태소 토큰화 (Morpheme Tokenization)
okt = Okt()
korean_morphs = okt.morphs(korean_text)
print("\n--- 3. 한국어 형태소 토큰화 ---")
print(korean_morphs)

# ④ 서브워드 토큰화 (Subword Tokenization - KoBERT Tokenizer)
tokenizer_ko = AutoTokenizer.from_pretrained("klue/roberta-base")

korean_text = "자연어 처리는 너무 재밌어요!"
korean_subwords = tokenizer_ko.tokenize(korean_text)

print("\n--- 4. 한국어 서브워드 토큰화 (KLUE RoBERTa) ---")
print(korean_subwords)

d:\SK_encoa\DL_WORKSPACE\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--- 1. 한국어 문장 토큰화 ---
['자연어 처리는 정말 재미있습니다.', '우리는 서브워드 토큰화를 배웁니다!']

--- 2. 한국어 단어(어절) 토큰화 ---
['자연어', '처리는', '정말', '재미있습니다.', '우리는', '서브워드', '토큰화를', '배웁니다!']

--- 3. 한국어 형태소 토큰화 ---
['자연어', '처리', '는', '정말', '재미있습니다', '.', '우리', '는', '서브', '워드', '토큰', '화', '를', '배웁니다', '!']



--- 4. 한국어 서브워드 토큰화 (KLUE RoBERTa) ---
['자연', '##어', '처리', '##는', '너무', '재밌', '##어요', '!']


In [1]:
from mecab import MeCab

mecab = MeCab()
print(mecab.pos("한국어 형태소 분석 테스트"))


ModuleNotFoundError: No module named 'mecab'

In [2]:
import time
from konlpy.tag import  Okt, Komoran, Kkma, Hannanum

import mecab_ko


class Mecab:
    def __init__(self):
        self.tagger = mecab_ko.Tagger()

    def pos(self, text):
        result = []

        for line in self.tagger.parse(text).splitlines():
            if line == "EOS" or "\t" not in line:
                continue

            word, features = line.split("\t", 1)
            tag = features.split(",", 1)[0]
            result.append((word, tag))

        return result

    def morphs(self, text):
        return [word for word, _ in self.pos(text)]

    def nouns(self, text):
        return [
            word
            for word, tag in self.pos(text)
            if tag.startswith("N")
        ]

# 테스트용 텍스트 (특수문자, 오탈자, 구어체, 명사 결합 포함)
sample_text = "KoNLPy는 한국어 자연어 처리를 위한 파이썬 라이브러리입니다. ㅋㅋㅋ 성능을 빠르게 비교해보세요!"

# 5개 분석기 인스턴스 초기화
analyzers = {
    'Mecab': Mecab(),
    'Okt': Okt(),
    'Komoran': Komoran(),
    'Kkma': Kkma(),
    'Hannanum': Hannanum()
}

# Mecab 설치 여부 예외 처리
# try:
#     analyzers['Mecab'] = Mecab()
# except Exception as e:
#     print("[알림] Mecab이 설치되어 있지 않아 Mecab 테스트는 제외됩니다.\n")

print("=" * 80)
print(f"테스트 문장: {sample_text}")
print("=" * 80)

for name, analyzer in analyzers.items():
    if analyzer is None:
        continue

    print(f"\n[ {name} 분석기 ]")

    # 1. 처리 속도 측정 (100회 반복 실행)
    start_time = time.time()
    for _ in range(100):
        analyzer.pos(sample_text)
    elapsed_time = (time.time() - start_time) / 100 * 1000  # ms 단위 변환

    # 2. 형태소 추출 (.morphs)
    morphs = analyzer.morphs(sample_text)

    # 3. 명사 추출 (.nouns)
    nouns = analyzer.nouns(sample_text)

    # 4. 품사 태깅 (.pos) - 앞부분 일부만 출력
    pos = analyzer.pos(sample_text)

    print(f" - 평균 실행 시간: {elapsed_time:.3f} ms")
    print(f" - 형태소 분할 결과: {morphs}")
    print(f" - 명사 추출 결과  : {nouns}")
    print(f" - 품사 태깅 예시  : {pos[:5]} ...")

테스트 문장: KoNLPy는 한국어 자연어 처리를 위한 파이썬 라이브러리입니다. ㅋㅋㅋ 성능을 빠르게 비교해보세요!

[ Mecab 분석기 ]
 - 평균 실행 시간: 0.713 ms
 - 형태소 분할 결과: ['KoNLPy', '는', '한국어', '자연어', '처리', '를', '위한', '파이썬', '라이브러리', '입니다', '.', 'ㅋㅋㅋ', '성능', '을', '빠르', '게', '비교', '해', '보', '세요', '!']
 - 명사 추출 결과  : ['한국어', '자연어', '처리', '파이썬', '라이브러리', '성능', '비교']
 - 품사 태깅 예시  : [('KoNLPy', 'SL'), ('는', 'JX'), ('한국어', 'NNG'), ('자연어', 'NNG'), ('처리', 'NNG')] ...

[ Okt 분석기 ]
 - 평균 실행 시간: 28.878 ms
 - 형태소 분할 결과: ['KoNLPy', '는', '한국어', '자연어', '처리', '를', '위', '한', '파이썬', '라이브러리', '입니다', '.', 'ㅋㅋㅋ', '성능', '을', '빠르게', '비교', '해보세요', '!']
 - 명사 추출 결과  : ['한국어', '자연어', '처리', '위', '파이썬', '라이브러리', '성능', '비교']
 - 품사 태깅 예시  : [('KoNLPy', 'Alpha'), ('는', 'Verb'), ('한국어', 'Noun'), ('자연어', 'Noun'), ('처리', 'Noun')] ...

[ Komoran 분석기 ]
 - 평균 실행 시간: 1.464 ms
 - 형태소 분할 결과: ['KoNLPy', '는', '한국어', '자연어', '처리', '를', '위하', 'ㄴ', '파이썬', '라이브러리', '이', 'ㅂ니다', '.', 'ㅋㅋㅋ', '성능', '을', '빠르', '게', '비교', '하', '아', '보', '시', '어요', '!']
 - 명사 추출 결과  : ['한국어', '자연어', '처리', '파이

In [3]:
import kss
from konlpy.tag import Okt

# Sample Data (마침표와 띄어쓰기가 다소 불완전한 문장)
raw_text = "자연어처리는 정말 재밌어요.KoNLPy와 KSS를 사용해봅시다! 문장 분리가 잘 될까요?"

print("=== 1. KSS를 이용한 문장 토큰화 ===")
sentences = kss.split_sentences(raw_text)
for idx, sent in enumerate(sentences, 1):
    print(f"문장 {idx}: {sent}")

print("\n=== 2. KoNLPy(Okt)를 이용한 형태소/명사 추출 ===")
okt = Okt()

for sent in sentences:
    # 형태소 추출
    morphs = okt.morphs(sent)
    # 명사만 추출
    nouns = okt.nouns(sent)
    
    print(f"\n원문: {sent}")
    print(f"형태소: {morphs}")
    print(f"명사: {nouns}")

c:\SKN35_kim\DL_WORKSPACE\.venv\Lib\site-packages\tossi\particles.py:243: SyntaxWarning: invalid escape sequence '\('
  I_PATTERN = re.compile(u'^이|\(이\)')
c:\SKN35_kim\DL_WORKSPACE\.venv\Lib\site-packages\jamo\jamo.py:79: SyntaxWarning: invalid escape sequence '\w'
  hcj_name = re.sub("(?<=HANGUL )(\w+)",
c:\SKN35_kim\DL_WORKSPACE\.venv\Lib\site-packages\jamo\jamo.py:210: SyntaxWarning: invalid escape sequence '\w'
  jamo_name = re.sub("(?<=HANGUL )(\w+)",
c:\SKN35_kim\DL_WORKSPACE\.venv\Lib\site-packages\whoosh\analysis\filters.py:56: SyntaxWarning: invalid escape sequence '\w'
  \w+([:.]?\w+)*         # word characters, with opt. internal colons/dots
c:\SKN35_kim\DL_WORKSPACE\.venv\Lib\site-packages\whoosh\analysis\filters.py:158: SyntaxWarning: invalid escape sequence '\S'
  >>> ana = RegexTokenizer(r"\S+") | TeeFilter(f1, f2)
c:\SKN35_kim\DL_WORKSPACE\.venv\Lib\site-packages\whoosh\analysis\intraword.py:49: SyntaxWarning: invalid escape sequence '\S'
  >>> analyzer = RegexTokenize

=== 1. KSS를 이용한 문장 토큰화 ===
문장 1: 자연어처리는 정말 재밌어요.
문장 2: KoNLPy와 KSS를 사용해봅시다!
문장 3: 문장 분리가 잘 될까요?

=== 2. KoNLPy(Okt)를 이용한 형태소/명사 추출 ===

원문: 자연어처리는 정말 재밌어요.
형태소: ['자연어', '처리', '는', '정말', '재밌어요', '.']
명사: ['자연어', '처리', '정말']

원문: KoNLPy와 KSS를 사용해봅시다!
형태소: ['KoNLPy', '와', 'KSS', '를', '사용', '해봅시다', '!']
명사: ['를', '사용']

원문: 문장 분리가 잘 될까요?
형태소: ['문장', '분리', '가', '잘', '될까', '요', '?']
명사: ['문장', '분리', '요']
